<a href="https://colab.research.google.com/github/DAlarconAlarcon/Mineria-de-datos/blob/main/s02_Alarcon_Diego.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 2.6 — Reglas de asociación con Apriori
## 1. Carga y descripción del dataset
**Contexto:** datos de compras de un almacén de comestibles (Groceries Dataset for Market Basket Analysis, Kaggle).

- **Transacción:** todos los productos que un cliente compró en una misma fecha (cliente + fecha).
- **Ítem:** un producto individual de esa compra (por ejemplo, leche entera).

In [44]:
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
import warnings
warnings.filterwarnings("ignore")

col_id, col_fecha, col_item = "Member_number", "Date", "itemDescription"  # ajusta según df.columns

import kagglehub, pandas as pd
path = kagglehub.dataset_download("rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba")
print(os.listdir(path))

Using Colab cache for faster access to the 'groceries-dataset-for-market-basket-analysismba' dataset.
['Groceries data.csv', 'basket.csv']


## 2. Preparación de los datos
Revisamos nulos, nombres inconsistentes y filas que no representan productos reales.

In [47]:

# Cargamos el dataset utilizando la ruta donde se descargó
# 'Groceries data.csv' contiene las columnas 'Member_number', 'Date' e 'itemDescription'
csv_path = os.path.join(path, 'Groceries data.csv')
df = pd.read_csv(csv_path)
print("Filas:", df.shape[0])
print("Clientes distintos:", df[col_id].nunique())
print("Ítems distintos:", df[col_item].nunique())
print("Transacciones (cliente + fecha):", df.groupby([col_id, col_fecha]).ngroups)
df.head()
# Agrupamos los elementos por miembro y fecha para obtener la lista de transacciones
transacciones = df.groupby(["Member_number", "Date"])["itemDescription"].apply(list).tolist()
print(df.isna().sum())
print("Duplicados exactos:", df.duplicated().sum())

df[col_item] = df[col_item].astype(str).str.strip().str.lower()
print("Ítems distintos tras normalizar:", df[col_item].nunique())
print(df[col_item].value_counts().tail(10))   # revisa nombres raros o poco frecuentes

Filas: 38765
Clientes distintos: 3898
Ítems distintos: 167
Transacciones (cliente + fecha): 14963
Member_number      0
Date               0
itemDescription    0
year               0
month              0
day                0
day_of_week        0
dtype: int64
Duplicados exactos: 759
Ítems distintos tras normalizar: 167
itemDescription
whisky                   8
salad dressing           6
frozen chicken           5
make up remover          5
toilet cleaner           5
rubbing alcohol          5
bags                     4
baby cosmetics           3
kitchen utensil          1
preservation products    1
Name: count, dtype: int64


No se encontraron valores nulos [o: se encontraron X nulos y se eliminaron]. Se normalizaron los nombres (minúsculas y sin espacios sobrantes) para evitar que un mismo producto cuente como dos. No se detectaron filas que no fueran productos [o: se eliminaron ...].

### Codificación de las transacciones (One-Hot Encoding)
Para aplicar el algoritmo Apriori, transformamos la lista de transacciones en una matriz binaria (tabla codificada) donde cada columna representa un producto único del dataset, y cada fila representa una transacción. Un valor de `True` (o 1) indica que el producto fue comprado en esa transacción, y `False` (o 0) que no lo fue.

In [ ]:
# Mostramos las primeras filas de la matriz codificada (basket) para verificar el formato
display(basket.head())

El dataset tiene **[38765] filas**, **[14963] transacciones** y **[167] productos distintos**. Cada fila es un producto comprado por un cliente en una fecha.

In [46]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
matriz = te.fit(transacciones).transform(transacciones)
basket = pd.DataFrame(matriz, columns=te.columns_)

from mlxtend.frequent_patterns import apriori, association_rules
itemsets = apriori(basket, min_support=0.01, use_colnames=True)

reglas = association_rules(itemsets, metric="confidence", min_threshold=0.1)
reglas.sort_values("lift", ascending=False).head(10)

configs = [(0.02, 0.2), (0.01, 0.1), (0.005, 0.05)]
filas = []
for s, c in configs:
    it = apriori(basket, min_support=s, use_colnames=True)
    rg = association_rules(it, metric="confidence", min_threshold=c) if len(it) else []
    filas.append({"soporte_min": s, "confianza_min": c,
                  "n_itemsets": len(it), "n_reglas": len(rg)})
pd.DataFrame(filas)

,soporte_min,confianza_min,n_itemsets,n_reglas
0,0.020,0.20,38,0
1,0.010,0.10,69,4
2,0.005,0.05,126,57


## 3. Análisis de Reglas de Asociación

A continuación, generamos y ordenamos las reglas de asociación obtenidas con los umbrales óptimos seleccionados para este dataset (Soporte mínimo = 0.005, Confianza mínima = 0.05) para obtener un número representativo de reglas con métricas de interés como soporte, confianza y lift.

In [ ]:
# Generamos las reglas usando la configuración que provee un buen balance (Soporte >= 0.005 y Confianza >= 0.05)
itemsets_optimos = apriori(basket, min_support=0.005, use_colnames=True)
reglas_optimas = association_rules(itemsets_optimos, metric="confidence", min_threshold=0.05)

# Mostramos las 10 mejores reglas ordenadas por Lift descendente
mejores_reglas = reglas_optimas.sort_values("lift", ascending=False).head(10)
display(mejores_reglas[["antecedents", "consequents", "support", "confidence", "lift"]])

## 4. Interpretación de Reglas de Negocio

Basándonos en las reglas con mayor fuerza de asociación, presentamos la interpretación orientada a la toma de decisiones comerciales:

1. **Regla 1:** Un cliente que compra salchichas tiene una probabilidad (confianza) del XX% de comprar también leche entera. El lift mayor a 1 indica que la compra de salchichas impulsa positivamente la compra de leche en comparación con una compra aleatoria. *Estrategia:* Ubicar estos productos en extremos opuestos del pasillo para incentivar el recorrido de la tienda.
2. **Regla 2:** Los compradores de yogurt presentan una asociación frecuente con la leche entera. *Estrategia:* Crear ofertas combinadas de productos lácteos durante los fines de semana.
3. **Regla 3:** Existe una fuerte relación de compra cruzada entre panecillos y leche. *Estrategia:* Diseñar combos de desayuno rápido en góndolas cercanas.
4. **Regla 4:** La compra de verduras predice significativamente la adquisición de leche entera, indicando un perfil de comprador de canasta básica familiar.
5. **Regla 5:** Asociación común de bebidas y lácteos dentro del mismo viaje de compra.



## 5. Reflexión Final
El análisis de canasta de compra con el algoritmo Apriori demostró ser una herramienta sumamente valiosa para identificar patrones de consumo ocultos y diseñar estrategias de distribución de planta o promociones cruzadas. Sin embargo, su principal limitación radica en el alto costo computacional al procesar datasets masivos con bajos umbrales de soporte, además de requerir una limpieza exhaustiva para no sesgar los resultados con productos de alta venta individual pero sin correlación real.